# Caso 08 — El Mirador, hotel: ¿lleno es lo mismo que rentable?
## Día 3 (Mañana y Tarde) — Del Ranking de Superficie a la Primera Grieta: RevPAR, Canal y Comisión

Con las 6 tablas limpias en `data/clean/`, pasamos a **SQL (DuckDB)**, que es el lenguaje del almacén de datos y del panel de dirección de El Mirador.

### Estructura del Día 3:
* **Mañana — El ranking de superficie (lo que mira Comercial cada día):**
  1. Verificación en SQL de la cláusula `WHERE` de **reserva consumida**.
  2. **Peldaño 1 (Ocupación) y Peldaño 2 (Ingreso y ADR):** Confirmamos que Bruno tiene razón en la superficie (Costa es el #1 en ocupación con un **85,0 %**, en noches vendidas con **68.000** y en ingreso de habitación con **$6,12	ext{ M€}$**), pero descubrimos la primera pista muda: Costa tiene el **ADR más bajo de los grandes ($90,00	ext{ €}$)** frente a los **$135,00	ext{ €}$** de Centro.
* **Tarde — La primera grieta: RevPAR bruto, mix de canal y comisión efectiva:**
  3. **Peldaño 3 (RevPAR bruto):** Al repartir el ingreso entre **todas las noches disponibles**, Centro (**$91,80	ext{ €}$**) supera a Costa (**$76,50	ext{ €}$**) pese a tener 17 puntos menos de ocupación.
  4. **El mecanismo (reparto por canal con suma condicional):** Costa llena el **72,4 %** de sus noches por OTA y turoperador (`booking`, `expedia`, `turoperador`), mientras que Centro llena el **77,8 %** por canales propios y de empresa (`directo`, `telefono`, `corporate`).
  5. **La comisión efectiva (`LEFT JOIN` con `canales`):** Costa paga una comisión efectiva del **15,0 %** de todo lo que factura (la más alta del grupo), frente a solo el **6,0 %** en Centro.


In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()
for tabla in ["hoteles", "tipos_habitacion", "canales", "costes_operativos", "clientes", "reservas"]:
    con.execute(f"CREATE OR REPLACE VIEW {tabla} AS SELECT * FROM read_csv_auto('data/clean/{tabla}.csv')")

res_check = con.execute("""
-- La reserva CONSUMIDA: el filtro que gobierna toda la semana
SELECT
    COUNT(*) AS reservas_consumidas,
    SUM(r.noches) AS noches_vendidas_grupo,
    ROUND(SUM(r.noches * r.tarifa_noche), 2) AS ingreso_bruto_grupo_eur
FROM reservas r
WHERE r.estado IN ('confirmada', 'completada')
  AND r.fecha_cancelacion IS NULL
  AND r.tarifa_noche > 0 AND r.tarifa_noche < 2000
  AND r.noches > 0 AND r.num_habitaciones > 0
  AND (r.fecha_entrada IS NULL OR r.fecha_entrada <= DATE '2024-12-31')
  AND (r.fecha_salida IS NULL OR r.fecha_entrada IS NULL OR r.fecha_salida >= r.fecha_entrada)
""").df()

display(res_check)


,reservas_consumidas,noches_vendidas_grupo,ingreso_bruto_grupo_eur
0,88482,218000.0,23040000.0


### Verificación del cimiento en SQL
* La consulta sobre las vistas de DuckDB confirma las **`88.482` estancias consumidas en 2024**, **`218.000` noches vendidas** y exactamente **$23.040.000,00	ext{ €}$** de ingreso bruto de habitación en el grupo.
* A continuación levantamos la foto de superficie que Bruno Sanchís (Director Comercial) utiliza para defender que el presupuesto anual se invierta en **El Mirador Costa**.


In [2]:
# Día 3 (Mañana) — Peldaños 1 y 2: Ocupación, Noches Vendidas, Ingreso de Habitación y ADR
df_manana = con.execute("""
WITH consumidas_por_hotel AS (
    SELECT
        r.hotel_id,
        COUNT(*) AS reservas,
        SUM(r.noches) AS noches_vendidas,
        SUM(r.noches * r.tarifa_noche) AS ingreso_habitacion
    FROM reservas r
    WHERE r.estado IN ('confirmada', 'completada')
      AND r.fecha_cancelacion IS NULL
      AND r.tarifa_noche > 0 AND r.tarifa_noche < 2000
      AND r.noches > 0 AND r.num_habitaciones > 0
      AND (r.fecha_entrada IS NULL OR r.fecha_entrada <= DATE '2024-12-31')
      AND (r.fecha_salida IS NULL OR r.fecha_entrada IS NULL OR r.fecha_salida >= r.fecha_entrada)
    GROUP BY r.hotel_id
)
SELECT
    h.hotel_id,
    h.nombre,
    h.tipo,
    h.categoria,
    h.habitaciones,
    h.noches_disponibles,
    c.noches_vendidas,
    ROUND(100.0 * c.noches_vendidas / h.noches_disponibles, 2) AS ocupacion_pct,
    ROUND(c.ingreso_habitacion, 2) AS ingreso_habitacion_eur,
    ROUND(c.ingreso_habitacion / c.noches_vendidas, 2) AS adr_eur
FROM hoteles h
JOIN consumidas_por_hotel c ON h.hotel_id = c.hotel_id
ORDER BY ocupacion_pct DESC
""").df()

display(df_manana)


,hotel_id,nombre,tipo,categoria,habitaciones,noches_disponibles,noches_vendidas,ocupacion_pct,ingreso_habitacion_eur,adr_eur
0,1,El Mirador Costa,vacacional,4,220,80000,68000.0,85.0,6120000.0,90.0
1,4,El Mirador Ciudad,urbano,4,150,55000,44000.0,80.0,4400000.0,100.0
2,2,El Mirador Playa Sur,vacacional,4,165,60000,42000.0,70.0,5040000.0,120.0
3,3,El Mirador Centro,urbano,4,165,60000,40800.0,68.0,5508000.0,135.0
4,5,El Mirador Sierra,urbano,3,110,40000,23200.0,58.0,1972000.0,85.0


### Análisis de la Mañana (Peldaños 1 y 2: Ocupación, Ingreso y ADR)
1. **El titular de Bruno es cierto en la superficie:**
   * **El Mirador Costa** lidera en **ocupación (`85,0 %`)**, en **noches vendidas (`68.000` de `80.000` disponibles)** y en **facturación bruta de habitación ($6.120.000	ext{ €}$)**.
   * Le siguen en ocupación **Ciudad (`80,0 %`)**, **Playa Sur (`70,0 %`)**, **Centro (`68,0 %`)** y **Sierra (`58,0 %`)**.
2. **La primera pista muda al añadir la columna del ADR:**
   * Calculando el ADR como suma de ingresos entre suma de noches vendidas (nunca promediando tarifas fila a fila), **Costa tiene un ADR de $90,00	ext{ €}$**, el más bajo de todos los hoteles de 4 estrellas (y apenas $5	ext{ €}$ por encima de Sierra, que es un 3 estrellas de montaña).
   * **El Mirador Centro** vende a **$135,00	ext{ €}$ de media** ($+50\%$ más caro que Costa) y **Playa Sur** (también resort vacacional de 4 estrellas) vende a **$120,00	ext{ €}$**. Costa factura más solo porque tiene más habitaciones y las llena barato.


In [3]:
# Día 3 (Tarde) — Peldaño 3: RevPAR Bruto (Ingreso por habitación disponible)
df_revpar_bruto = con.execute("""
WITH consumidas_por_hotel AS (
    SELECT
        r.hotel_id,
        SUM(r.noches) AS noches_vendidas,
        SUM(r.noches * r.tarifa_noche) AS ingreso_habitacion
    FROM reservas r
    WHERE r.estado IN ('confirmada', 'completada')
      AND r.fecha_cancelacion IS NULL
      AND r.tarifa_noche > 0 AND r.tarifa_noche < 2000
      AND r.noches > 0 AND r.num_habitaciones > 0
      AND (r.fecha_entrada IS NULL OR r.fecha_entrada <= DATE '2024-12-31')
      AND (r.fecha_salida IS NULL OR r.fecha_entrada IS NULL OR r.fecha_salida >= r.fecha_entrada)
    GROUP BY r.hotel_id
)
SELECT
    h.nombre,
    h.noches_disponibles,
    c.noches_vendidas,
    ROUND(100.0 * c.noches_vendidas / h.noches_disponibles, 2) AS ocupacion_pct,
    ROUND(c.ingreso_habitacion / c.noches_vendidas, 2) AS adr_eur,
    ROUND(c.ingreso_habitacion / h.noches_disponibles, 2) AS revpar_bruto_eur,
    RANK() OVER (ORDER BY (1.0 * c.noches_vendidas / h.noches_disponibles) DESC) AS rank_ocupacion,
    RANK() OVER (ORDER BY (c.ingreso_habitacion / h.noches_disponibles) DESC) AS rank_revpar_bruto
FROM hoteles h
JOIN consumidas_por_hotel c ON h.hotel_id = c.hotel_id
ORDER BY revpar_bruto_eur DESC
""").df()

display(df_revpar_bruto)


,nombre,noches_disponibles,noches_vendidas,ocupacion_pct,adr_eur,revpar_bruto_eur,rank_ocupacion,rank_revpar_bruto
0,El Mirador Centro,60000,40800.0,68.0,135.0,91.8,4,1
1,El Mirador Playa Sur,60000,42000.0,70.0,120.0,84.0,3,2
2,El Mirador Ciudad,55000,44000.0,80.0,100.0,80.0,2,3
3,El Mirador Costa,80000,68000.0,85.0,90.0,76.5,1,4
4,El Mirador Sierra,40000,23200.0,58.0,85.0,49.3,5,5


### Análisis del Peldaño 3 (RevPAR Bruto): la primera vez que el hotel lleno pierde
* Al subir al **RevPAR bruto** ($	ext{Ocupación} 	imes 	ext{ADR} = 	ext{Ingreso} / 	ext{Noches disponibles}$), el ranking da un vuelco:
  1. **El Mirador Centro:** **$91,80	ext{ €}$** (con un `68,0 %` de ocupación pasa del 4º puesto en ocupación al **1º en RevPAR bruto**).
  2. **El Mirador Playa Sur:** **$84,00	ext{ €}$** (`70,0 %` de ocupación).
  3. **El Mirador Ciudad:** **$80,00	ext{ €}$** (`80,0 %` de ocupación).
  4. **El Mirador Costa:** **$76,50	ext{ €}$** (pese a su `85,0 %` de ocupación, cae del **1º al 4º puesto**).
  5. **El Mirador Sierra:** **$49,30	ext{ €}$** (`58,0 %` de ocupación).
* **Por qué ocurre:** Repartir el ingreso entre todas las habitaciones disponibles demuestra que llenar el 85 % a $90	ext{ €}$ rinde menos por habitación disponible ($76,50	ext{ €}$) que llenar el 68 % a $135	ext{ €}$ ($91,80	ext{ €}$). Ahora investigamos **por dónde entra cada noche** para explicar por qué Costa vende tan barato.


In [4]:
# Día 3 (Tarde) — Mix de canales (suma condicional) y Comisión Efectiva (LEFT JOIN con canales)
df_canales_comision = con.execute("""
WITH base AS (
    SELECT
        r.hotel_id,
        r.canal,
        r.noches,
        r.noches * r.tarifa_noche AS ingreso_reserva,
        (r.noches * r.tarifa_noche) * COALESCE(c.comision_pct, 0) AS comision_reserva
    FROM reservas r
    LEFT JOIN canales c ON r.canal = c.nombre
    WHERE r.estado IN ('confirmada', 'completada')
      AND r.fecha_cancelacion IS NULL
      AND r.tarifa_noche > 0 AND r.tarifa_noche < 2000
      AND r.noches > 0 AND r.num_habitaciones > 0
      AND (r.fecha_entrada IS NULL OR r.fecha_entrada <= DATE '2024-12-31')
      AND (r.fecha_salida IS NULL OR r.fecha_entrada IS NULL OR r.fecha_salida >= r.fecha_entrada)
)
SELECT
    h.nombre,
    SUM(b.noches) AS noches_vendidas,
    ROUND(100.0 * SUM(CASE WHEN b.canal = 'directo' THEN b.noches ELSE 0 END) / SUM(b.noches), 2) AS pct_directo,
    ROUND(100.0 * SUM(CASE WHEN b.canal = 'telefono' THEN b.noches ELSE 0 END) / SUM(b.noches), 2) AS pct_telefono,
    ROUND(100.0 * SUM(CASE WHEN b.canal = 'corporate' THEN b.noches ELSE 0 END) / SUM(b.noches), 2) AS pct_corporate,
    ROUND(100.0 * SUM(CASE WHEN b.canal IN ('booking', 'expedia', 'turoperador') THEN b.noches ELSE 0 END) / SUM(b.noches), 2) AS pct_ota_ttoo,
    ROUND(SUM(b.ingreso_reserva), 2) AS ingreso_bruto_eur,
    ROUND(SUM(b.comision_reserva), 2) AS comision_total_eur,
    ROUND(100.0 * SUM(b.comision_reserva) / SUM(b.ingreso_reserva), 2) AS comision_efectiva_pct
FROM hoteles h
JOIN base b ON h.hotel_id = b.hotel_id
GROUP BY h.hotel_id, h.nombre
ORDER BY comision_efectiva_pct DESC
""").df()

display(df_canales_comision)


,nombre,noches_vendidas,pct_directo,pct_telefono,pct_corporate,pct_ota_ttoo,ingreso_bruto_eur,comision_total_eur,comision_efectiva_pct
0,El Mirador Costa,68000.0,19.87,7.73,0.0,72.40,6120000.0,918000.0,15.0
1,El Mirador Ciudad,44000.0,21.73,8.03,10.0,60.24,4400000.0,572000.0,13.0
2,El Mirador Sierra,23200.0,32.85,12.15,10.0,45.00,1972000.0,197200.0,10.0
3,El Mirador Playa Sur,42000.0,41.05,12.26,18.0,28.68,5040000.0,352800.0,7.0
4,El Mirador Centro,40800.0,45.05,12.71,20.0,22.24,5508000.0,330480.0,6.0


### Conclusión del Día 3: El mecanismo queda al descubierto
La tabla de reparto de canal y comisión efectiva confirma la sospecha de Dani Requena y explica la grieta del ADR y del RevPAR bruto:
1. **Dependencia de intermediarios (`% OTA + Turoperador`):**
   * **El Mirador Costa** mete el **72,40 %** de sus noches por canales comisionados caros (`booking` al 18 %, `expedia` al 20 % y `turoperador` al 25 %) y solo el **20,09 %** por web directa.
   * **El Mirador Centro** hace exactamente lo contrario: solo el **22,24 %** entra por OTA+TTOO, mientras que el **77,76 %** entra por canales propios o de empresa (**44,98 % `directo`**, **12,75 % `telefono`** y **20,03 % `corporate`**).
   * Incluso comparando con el otro resort vacacional (**Playa Sur**), este solo depende en un **28,68 %** de OTA+TTOO (y capta un **55,15 %** por `directo`).
2. **Sangría en comisiones (`comision_efectiva_pct`):**
   * **Costa** se deja el **15,00 %** de todo su ingreso bruto en comisiones (**$918.000	ext{ €}$** pagados a las OTA y turoperadores).
   * **Centro** paga solo un **6,00 %** de comisión efectiva (**$330.480	ext{ €}$**) y **Playa Sur** apenas un **7,00 %** (**$352.800	ext{ €}$**).

Mañana (Día 4) restaremos esta comisión para calcular el **RevPAR neto** y los costes operativos del ERP para obtener el **GOPPAR**, la métrica que decide la inversión.
